# Example 2: Receding Horizon Control for a Simple System

We compare three controllers on the same discrete-time double-integrator system:

1. Infinite-horizon LQR (traditional LQR)
2. Finite-horizon LQR with no terminal cost
3. Finite-horizon LQR with terminal cost

For each method, we show a **single-state timeline**:
- Solid line: realized closed-loop trajectory
- Dot: current state
- Dotted line: predicted trajectory from current time (receding horizon prediction)

We also report compute cost per method.

In [7]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are
import ipywidgets as widgets
from IPython.display import display

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "font.size": 11,
    }
)

In [8]:
# Discrete-time double integrator
Ts = 0.2
Ad = np.array([[1.0, Ts], [0.0, 1.0]])
Bd = np.array([[0.5 * Ts**2], [Ts]])

nx, nu = Bd.shape[0], Bd.shape[1]

Q = np.diag([4.0, 0.5])
R = np.array([[0.2]])

x0 = np.array([4.0, 0.0])
T_sim = 40
N_pred = 10
plot_state_idx = 0
plot_state_label = "x1 (position)"

print("Ad =\n", Ad)
print("Bd =\n", Bd)
print(f"Simulation steps: {T_sim}, prediction horizon: {N_pred}")

Ad =
 [[1.  0.2]
 [0.  1. ]]
Bd =
 [[0.02]
 [0.2 ]]
Simulation steps: 40, prediction horizon: 10


In [9]:
def compute_infinite_horizon_gain(Ad, Bd, Q, R):
    t0 = time.perf_counter()
    P_inf = solve_discrete_are(Ad, Bd, Q, R)
    K_inf = np.linalg.solve(R + Bd.T @ P_inf @ Bd, Bd.T @ P_inf @ Ad)
    t_ms = (time.perf_counter() - t0) * 1000.0
    return K_inf, P_inf, t_ms


def finite_horizon_gains(Ad, Bd, Q, R, N, P_terminal):
    P = [None] * (N + 1)
    K = [None] * N
    P[N] = P_terminal.copy()
    for k in range(N - 1, -1, -1):
        S = R + Bd.T @ P[k + 1] @ Bd
        K[k] = np.linalg.solve(S, Bd.T @ P[k + 1] @ Ad)
        P[k] = Q + Ad.T @ P[k + 1] @ Ad - Ad.T @ P[k + 1] @ Bd @ K[k]
    return K, P


def predict_trajectory(Ad, Bd, xk, K_policy):
    N = len(K_policy)
    x_pred = np.zeros((N + 1, Ad.shape[0]))
    u_pred = np.zeros((N, Bd.shape[1]))
    x_pred[0] = xk
    for j in range(N):
        uj = -(K_policy[j] @ x_pred[j])
        u_pred[j] = uj
        x_pred[j + 1] = Ad @ x_pred[j] + Bd @ uj
    return x_pred, u_pred


def simulate_controller(method_name, Ad, Bd, Q, R, x0, T_sim, N_pred, P_inf=None):
    x = np.zeros((T_sim + 1, Ad.shape[0]))
    u = np.zeros((T_sim, Bd.shape[1]))
    x_pred_store = np.zeros((T_sim + 1, N_pred + 1, Ad.shape[0]))
    step_ms = np.zeros(T_sim)

    x[0] = x0

    if method_name == "infinite":
        K_inf, P_inf_local, setup_ms = compute_infinite_horizon_gain(Ad, Bd, Q, R)
        if P_inf is None:
            P_inf = P_inf_local

    sim_t0 = time.perf_counter()

    for k in range(T_sim):
        t0 = time.perf_counter()

        if method_name == "infinite":
            K_policy = [K_inf for _ in range(N_pred)]
        elif method_name == "finite_no_terminal":
            K_policy, _ = finite_horizon_gains(Ad, Bd, Q, R, N_pred, np.zeros_like(Q))
            setup_ms = 0.0
        elif method_name == "finite_with_terminal":
            K_policy, _ = finite_horizon_gains(Ad, Bd, Q, R, N_pred, P_inf)
            setup_ms = 0.0
        else:
            raise ValueError("Unknown method")

        x_pred, u_pred = predict_trajectory(Ad, Bd, x[k], K_policy)
        uk = u_pred[0]

        step_ms[k] = (time.perf_counter() - t0) * 1000.0

        u[k] = uk
        x[k + 1] = Ad @ x[k] + Bd @ uk
        x_pred_store[k] = x_pred

    if method_name == "infinite":
        K_policy_last = [K_inf for _ in range(N_pred)]
    elif method_name == "finite_no_terminal":
        K_policy_last, _ = finite_horizon_gains(Ad, Bd, Q, R, N_pred, np.zeros_like(Q))
    else:
        K_policy_last, _ = finite_horizon_gains(Ad, Bd, Q, R, N_pred, P_inf)

    x_pred_store[T_sim], _ = predict_trajectory(Ad, Bd, x[T_sim], K_policy_last)

    total_ms = (time.perf_counter() - sim_t0) * 1000.0

    return {
        "x": x,
        "u": u,
        "x_pred": x_pred_store,
        "step_ms": step_ms,
        "setup_ms": setup_ms,
        "total_ms": total_ms,
    }

In [10]:
results = {}

results["Infinite-horizon LQR"] = simulate_controller(
    method_name="infinite",
    Ad=Ad,
    Bd=Bd,
    Q=Q,
    R=R,
    x0=x0,
    T_sim=T_sim,
    N_pred=N_pred,
)

P_inf = solve_discrete_are(Ad, Bd, Q, R)

results["Finite-horizon LQR (no terminal cost)"] = simulate_controller(
    method_name="finite_no_terminal",
    Ad=Ad,
    Bd=Bd,
    Q=Q,
    R=R,
    x0=x0,
    T_sim=T_sim,
    N_pred=N_pred,
    P_inf=P_inf,
)

results["Finite-horizon LQR (with terminal cost)"] = simulate_controller(
    method_name="finite_with_terminal",
    Ad=Ad,
    Bd=Bd,
    Q=Q,
    R=R,
    x0=x0,
    T_sim=T_sim,
    N_pred=N_pred,
    P_inf=P_inf,
)

summary_rows = []
for name, res in results.items():
    summary_rows.append(
        {
            "Method": name,
            "Setup cost [ms]": res["setup_ms"],
            "Mean step cost [ms]": float(np.mean(res["step_ms"])),
            "Max step cost [ms]": float(np.max(res["step_ms"])),
            "Total runtime [ms]": res["total_ms"],
        }
    )

cost_df = pd.DataFrame(summary_rows)
cost_df

,Method,Setup cost [ms],Mean step cost [ms],Max step cost [ms],Total runtime [ms]
0,Infinite-horizon LQR,0.7653,0.026247,0.0330,1.1613
1,Finite-horizon LQR (no terminal cost),0.0000,0.131545,0.1590,5.4808
2,Finite-horizon LQR (with terminal cost),0.0000,0.127443,0.1678,5.3180


In [11]:
method_names = list(results.keys())
t = np.arange(T_sim + 1) * Ts

all_vals = []
for name in method_names:
    all_vals.extend(results[name]["x"][:, plot_state_idx].tolist())
    all_vals.extend(results[name]["x_pred"][:, :, plot_state_idx].ravel().tolist())

ymin = min(all_vals)
ymax = max(all_vals)
pad = 0.08 * max(1e-6, ymax - ymin)

def plot_receding_horizon(k=0):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

    for ax, name in zip(axes, method_names):
        res = results[name]
        x_real = res["x"][:, plot_state_idx]
        x_pred = res["x_pred"][k, :, plot_state_idx]

        t_pred = t[k] + np.arange(N_pred + 1) * Ts

        ax.plot(t, x_real, color="C0", lw=2.0, label="Realized trajectory")
        ax.plot(t[k], x_real[k], "o", color="C0", ms=8, label="Current state")
        ax.plot(t_pred, x_pred, "--", color="C3", lw=2.0, label="Predicted trajectory")

        ax.set_title(name)
        ax.set_xlabel("Time [s]")
        ax.set_ylim(ymin - pad, ymax + pad)

    axes[0].set_ylabel(plot_state_label)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
    fig.suptitle(f"Single-state timeline with receding-horizon prediction (k={k})", y=1.15)
    fig.tight_layout()
    plt.show()

widgets.interact(
    plot_receding_horizon,
    k=widgets.IntSlider(value=0, min=0, max=T_sim, step=1, description="Current step"),
);

interactive(children=(IntSlider(value=0, description='Current step', max=40), Output()), _dom_classes=('widget…

### Compute-cost interpretation

- **Infinite-horizon LQR** computes one Riccati solution once (setup), then uses a fixed gain online.
- **Finite-horizon LQR (no terminal cost)** recomputes a backward Riccati recursion each control step.
- **Finite-horizon LQR (with terminal cost)** does the same online recursion, but uses $P_N = P_\infty$, often giving better short-horizon behavior.

This is why online compute cost is usually lowest for infinite-horizon LQR and higher for finite-horizon receding-horizon variants.